# Additional file 2 — PhysioNet/CinC Challenge 2012 analyses and measurement-error simulation

| Section | Paper |
|---|---|
| 3. Data preparation from the raw files | Methods |
| 5. Measurement-error diagnostics | Table 5 |
| 6. Synthetic measurement-error simulation | Fig. 4 |
| 7. Resampling experiment with full-cohort reference values, 1,000 replicates | Table 6, Fig. 2 (right) |
| 8. Finite configurations against a dense search (scenario B) | Table 4 |
| 9. Application with crisp and graded contours | Table 7, Figs. 5–6 |
| 10. Discrimination (AUC) | Fig. 7 |

The code is identical to the code that produced the paper's results, and so are the seeds (replicate *r* uses seed 50,000 + *r*).

**Data.** Download the challenge data from https://physionet.org/content/challenge-2012/ (ODC-BY 1.0): `set-a`, `set-b`, `set-c` and `Outcomes-a/b/c.txt`. Set `DATA_DIR`.

**Run time.** `QUICK = True`: about 10–20 minutes, including reading the 12,000 raw files twice. Full run: a few CPU-hours.

## 1. Settings

In [ ]:
DATA_DIR = "physionet-challenge-2012"
QUICK = True
OUTDIR = "results_p12"
COV_REPS = 5 if QUICK else 1000
DENSE_REPS = 2 if QUICK else 100
ME_REPS = 5 if QUICK else 250
CIDX_REPS = 2 if QUICK else 30
import os; os.makedirs(OUTDIR, exist_ok=True)

## 2. Core library

In [ ]:
"""
hppsm_v3.py - core library for the HPPSM analyses (Draft 7).

Contents
  * RoystonParmarPH : flexible parametric PH model (JAX likelihood), fitted by L-BFGS-B and then
                      refined by Newton-Raphson steps to a gradient norm below 1e-8, so the maximum
                      likelihood estimate does not depend on optimiser stopping rules.
  * logit_fit       : unpenalised logistic regression (coefficients and SEs per raw unit).
  * rubin           : Rubin's rules.
  * smcfcs_rp       : substantive-model-compatible FCS imputation for the RP model (Bartlett et al. 2015).
  * regression_calibration, error_variance_bound : interval-summarised covariates.
  * Envelope        : union of core intervals over mechanism configurations, computed by
                        - vertex_2q3  : reference, each variable at +/-dmax alone, all +dmax, all -dmax
                        - adaptive    : the above plus, for every endpoint of every target, the vertex
                                        whose signs are read off the single-variable runs (exact when
                                        each endpoint is monotone in each coordinate; Proposition 5)
                        - dense       : all 2^q vertices plus random points in the box (verification)
"""
import itertools
import numpy as np, pandas as pd
import jax, jax.numpy as jnp
from scipy.optimize import minimize
from scipy.stats import t as tdist
jax.config.update("jax_enable_x64", True)
Q95 = 1.959963984540054


# ------------------------------------------------------------------ Royston-Parmar model
def _standardise(A, mu=None, sd=None):
    A = np.asarray(A, float)
    if mu is None:
        mu = A.mean(0); sd = A.std(0); sd = np.where(sd > 0, sd, 1.0)
    return (A - mu) / sd, mu, sd


def _pos3(x):
    return jnp.where(x > 0, x, 0.0) ** 3


def _dpos3(x):
    return 3.0 * jnp.where(x > 0, x, 0.0) ** 2


def rp_basis(u, knots):
    kmin, kmax = knots[0], knots[-1]
    cols = [jnp.ones_like(u), u]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        cols.append(_pos3(u - k) - lam * _pos3(u - kmin) - (1 - lam) * _pos3(u - kmax))
    return jnp.stack(cols, axis=-1)


def rp_dbasis(u, knots):
    kmin, kmax = knots[0], knots[-1]
    cols = [jnp.zeros_like(u), jnp.ones_like(u)]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        cols.append(_dpos3(u - k) - lam * _dpos3(u - kmin) - (1 - lam) * _dpos3(u - kmax))
    return jnp.stack(cols, axis=-1)


def _rp_ll(th, u, d, Zs, Xs, knots, ng):
    g, bz, bx = th[:ng], th[ng:ng + Zs.shape[1]], th[ng + Zs.shape[1]:]
    s = rp_basis(u, knots) @ g; ds = rp_dbasis(u, knots) @ g
    eta = Zs @ bz + Xs @ bx
    return jnp.sum(d * (jnp.log(jnp.clip(ds, 1e-10)) - u + s + eta) - jnp.exp(s + eta))


_rp_nll = jax.jit(lambda *a: -_rp_ll(*a), static_argnums=6)
_rp_grad = jax.jit(jax.grad(lambda *a: -_rp_ll(*a)), static_argnums=6)
_rp_hess = jax.jit(jax.hessian(_rp_ll), static_argnums=6)


def _newton_polish(f, g, h, th, tol=1e-8, max_iter=8):
    """Newton-Raphson refinement of a minimiser of f (h = Hessian of -f) with step halving."""
    fv = f(th)
    for _ in range(max_iter):
        gr = g(th)
        if np.max(np.abs(gr)) < tol:
            break
        H = h(th)
        try:
            step = np.linalg.solve(H, gr)          # H is the log-likelihood Hessian (negative definite)
        except np.linalg.LinAlgError:
            break
        lam = 1.0
        while lam > 1e-4:
            new = th + lam * step
            fn = f(new)
            if np.isfinite(fn) and fn <= fv + 1e-12:
                th, fv = new, fn; break
            lam /= 2
        else:
            break
    return th


def _np_basis(u, knots):
    knots = np.asarray(knots); kmin, kmax = knots[0], knots[-1]
    p3 = lambda x: np.where(x > 0, x, 0.0) ** 3; d3 = lambda x: 3.0 * np.where(x > 0, x, 0.0) ** 2
    B = [np.ones_like(u), u]; dB = [np.zeros_like(u), np.ones_like(u)]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        B.append(p3(u - k) - lam * p3(u - kmin) - (1 - lam) * p3(u - kmax))
        dB.append(d3(u - k) - lam * d3(u - kmin) - (1 - lam) * d3(u - kmax))
    return np.column_stack(B), np.column_stack(dB)


class _RPLik:
    """Analytic log-likelihood, gradient and Hessian of the Royston-Parmar PH model (numpy)."""

    def __init__(self, u, d, Zs, Xs, knots):
        self.B, self.dB = _np_basis(u, knots); self.u, self.d = u, d
        self.D = np.hstack([self.B, Zs, Xs]); self.ng = self.B.shape[1]

    def _parts(self, th):
        ds = self.dB @ th[:self.ng]; lin = self.D @ th
        return ds, lin

    def nll(self, th):
        ds, lin = self._parts(th)
        if np.any(ds <= 0):
            return np.inf
        return -float(np.sum(self.d * (np.log(ds) - self.u + lin) - np.exp(np.minimum(lin, 700))))

    def grad(self, th):                       # gradient of the negative log-likelihood
        ds, lin = self._parts(th); mu = np.exp(np.minimum(lin, 700))
        g = self.D.T @ (self.d - mu)
        g[:self.ng] += self.dB.T @ (self.d / np.clip(ds, 1e-10, None))
        return -g

    def hess(self, th):                       # Hessian of the log-likelihood
        ds, lin = self._parts(th); mu = np.exp(np.minimum(lin, 700))
        H = -(self.D.T * mu) @ self.D
        w = self.d / np.clip(ds, 1e-10, None) ** 2
        H[:self.ng, :self.ng] -= (self.dB.T * w) @ self.dB
        return H


class RoystonParmarPH:
    """log H(t|z,x) = s(log t) + z'b_z + x'b_x; s a natural cubic spline with `df` d.f.
    Covariates are standardised internally (theta and cov on the standardised scale)."""

    def __init__(self, df=3):
        self.df = df

    def fit(self, t, event, Z, X, theta0=None):
        t = np.asarray(t, float); event = np.asarray(event, float); u = np.log(t)
        self.knots = jnp.asarray(np.quantile(u[event == 1], np.linspace(0, 1, self.df + 1)))
        Zs, self.z_mu, self.z_sd = _standardise(Z)
        Xs, self.x_mu, self.x_sd = _standardise(X)
        Zs, Xs = np.ascontiguousarray(Zs), np.ascontiguousarray(Xs)
        ng = self.df + 1
        self.ix_g = np.arange(ng); self.ix_z = np.arange(ng, ng + Zs.shape[1])
        self.ix_x = np.arange(ng + Zs.shape[1], ng + Zs.shape[1] + Xs.shape[1])
        warm = theta0 is not None
        if theta0 is None:
            theta0 = np.zeros(ng + Zs.shape[1] + Xs.shape[1]); theta0[0] = np.log(event.sum() / t.sum()); theta0[1] = 1.0
        L = _RPLik(u, event, Zs, Xs, np.asarray(self.knots))
        f, g, h = L.nll, L.grad, L.hess
        th = None
        if warm:                                   # warm start: Newton-Raphson directly (quadratic convergence)
            th = _newton_polish(f, g, h, np.asarray(theta0, float), max_iter=25)
            if not np.all(np.isfinite(th)) or np.max(np.abs(g(th))) > 1e-6:
                th = None
        if th is None:
            res = minimize(f, theta0, jac=g, method="L-BFGS-B", options={"maxiter": 5000, "gtol": 1e-7})
            th = _newton_polish(f, g, h, res.x)
        self.theta = th
        self.cov = np.linalg.inv(-h(self.theta))
        self.loglik = -f(self.theta)
        return self

    def coef(self, block, j):
        k = (self.ix_z if block == "z" else self.ix_x)[j]
        s = (self.z_sd if block == "z" else self.x_sd)[j]
        return self.theta[k] / s, np.sqrt(self.cov[k, k]) / s

    def lp(self, Z, X):
        return ((np.asarray(Z) - self.z_mu) / self.z_sd) @ self.theta[self.ix_z] + \
               ((np.asarray(X) - self.x_mu) / self.x_sd) @ self.theta[self.ix_x]


# ------------------------------------------------------------------ logistic regression
def logit_fit(y, X):
    """Unpenalised logistic regression by Newton-Raphson. Returns (coefficients, SEs) per raw unit
    (intercept excluded) and the intercept as a third element."""
    X = np.asarray(X, float); y = np.asarray(y, float)
    mu, s = X.mean(0), X.std(0); s = np.where(s > 0, s, 1.0)
    Xd = np.hstack([np.ones((len(y), 1)), (X - mu) / s])
    b = np.zeros(Xd.shape[1]); b[0] = np.log(y.mean() / (1 - y.mean()))
    for _ in range(50):
        eta = Xd @ b; p = 1 / (1 + np.exp(-eta))
        W = p * (1 - p); H = Xd.T @ (Xd * W[:, None]); gr = Xd.T @ (y - p)
        step = np.linalg.solve(H, gr); b = b + step
        if np.max(np.abs(step)) < 1e-10:
            break
    p = 1 / (1 + np.exp(-(Xd @ b)))
    cov = np.linalg.inv(Xd.T @ (Xd * (p * (1 - p))[:, None]))
    coef = b[1:] / s
    return coef, np.sqrt(np.diag(cov))[1:] / s, b[0] - np.sum(b[1:] * mu / s)


# ------------------------------------------------------------------ pooling
def rubin(ests, vars_, alpha=0.05):
    ests, vars_ = np.asarray(ests, float), np.asarray(vars_, float)
    m = len(ests); qb = ests.mean(0); W = vars_.mean(0)
    if m == 1:
        z = tdist.ppf(1 - alpha / 2, 1e6); return qb, qb - z * np.sqrt(W), qb + z * np.sqrt(W)
    B = ests.var(0, ddof=1); Tv = W + (1 + 1 / m) * B
    with np.errstate(divide="ignore", invalid="ignore"):
        dfr = np.where(B > 0, (m - 1) * (1 + W / ((1 + 1 / m) * B)) ** 2, 1e6)
    tq = tdist.ppf(1 - alpha / 2, dfr)
    return qb, qb - tq * np.sqrt(Tv), qb + tq * np.sqrt(Tv)


# ------------------------------------------------------------------ interval-summarised covariates
def error_variance_bound(lo, hi, n):
    """Bound (1): Var(U) <= (n-2) w^2 / (4 n^2), interior measurements treated as independent."""
    return np.maximum(n - 2, 0) * (hi - lo) ** 2 / (4 * np.maximum(n, 1) ** 2)


def regression_calibration(W, Z, sigma_u2):
    """E[X*|W,Z] under W = X* + U with U independent of (X*, Z, Y), E U = 0, Var U = diag(sigma_u2);
    computed from first and second moments. Error variance capped at 80% of Var(W)."""
    V = np.hstack([W, Z]); mu = V.mean(0); S = np.cov(V, rowvar=False)
    q = W.shape[1]
    sigma_u2 = np.minimum(sigma_u2, 0.8 * np.diag(S)[:q])
    C = S[:q, :].copy(); C[:, :q] -= np.diag(sigma_u2)
    return W.mean(0) + (V - mu) @ np.linalg.solve(S, C.T)


# ------------------------------------------------------------------ SMC-FCS for the RP model
def truncnorm_draw(rng, mu, sd, lo, hi):
    """Exact draws from N(mu, sd^2) truncated to [lo, hi] by inversion, computed in the tail that keeps
    precision (no clipping, so no artificial mass at the limits)."""
    from scipy.special import ndtr, ndtri
    mu = np.asarray(mu, float); a = (lo - mu) / sd; b = (hi - mu) / sd
    u = rng.uniform(size=mu.shape)
    x = np.empty_like(mu)
    up = a > 0                                     # whole interval in the upper tail: invert the survival function
    Ga, Gb = ndtr(-a[up]), ndtr(-b[up])
    x[up] = -ndtri(Gb + u[up] * (Ga - Gb))
    dn = ~up
    Fa, Fb = ndtr(a[dn]), ndtr(b[dn])
    x[dn] = ndtri(Fa + u[dn] * (Fb - Fa))
    return np.clip(mu + sd * x, lo, hi)            # clip only guards against rounding at the limits


def smcfcs_rp(T, E, Z, Xmiss, lo, hi, n_cycles=8, m=3, seed=0, max_tries=1000, theta0=None, return_diag=False):
    """Substantive-model-compatible FCS imputation (Bartlett et al. 2015) for the Royston-Parmar model.
    Proposals come from the normal linear covariate model (parameters drawn from their posterior) truncated
    to the variable's support [lo, hi]; they are accepted with probability exp(-H) if censored and
    H exp(1 - H) if died, with H the cumulative hazard under a parameter draw from the current model fit.
    Values not accepted after `max_tries` proposals keep their value from the previous cycle; the number of
    such values is reported in the diagnostics (return_diag=True)."""
    rng = np.random.default_rng(seed)
    Xm = np.asarray(Xmiss, float); M = np.isnan(Xm); n, q = Xm.shape
    miss_cols = [j for j in range(q) if M[:, j].any()]
    out = []; diag = dict(draws=0, not_accepted=0, proposals=0, final_cycle_not_accepted=0)
    for k in range(m):
        X = Xm.copy()
        for j in range(q):
            X[M[:, j], j] = np.nanmean(Xm[:, j])
        th_prev = theta0
        for cyc in range(n_cycles):
            rp = RoystonParmarPH(df=3).fit(T, E, Z, X, theta0=th_prev); th_prev = rp.theta
            th = rng.multivariate_normal(rp.theta, rp.cov)
            s = np.asarray(rp_basis(jnp.asarray(np.log(T)), rp.knots)) @ th[rp.ix_g]
            eta_z = ((Z - rp.z_mu) / rp.z_sd) @ th[rp.ix_z]
            bx = th[rp.ix_x] / rp.x_sd
            for j in miss_cols:
                rows = np.where(M[:, j])[0]
                D = np.hstack([np.ones((n, 1)), Z, np.delete(X, j, axis=1)])
                obs = ~M[:, j]; Do, yo = D[obs], X[obs, j]
                XtX = Do.T @ Do + 1e-8 * np.eye(D.shape[1])
                bh = np.linalg.solve(XtX, Do.T @ yo)
                res = yo - Do @ bh; dfree = max(obs.sum() - D.shape[1], 1)
                s2 = (res @ res) / rng.chisquare(dfree)
                bd = rng.multivariate_normal(bh, s2 * np.linalg.inv(XtX))
                mu = D[rows] @ bd
                eta_other = eta_z[rows] + (X[rows] - rp.x_mu) @ bx - (X[rows, j] - rp.x_mu[j]) * bx[j]
                pending = np.ones(len(rows), bool); draw = X[rows, j].copy()     # fall-back: previous value
                for _ in range(max_tries):
                    if not pending.any():
                        break
                    idx = np.where(pending)[0]
                    prop = truncnorm_draw(rng, mu[idx], np.sqrt(s2), lo[j], hi[j])
                    H = np.exp(np.minimum(s[rows[idx]] + eta_other[idx] + (prop - rp.x_mu[j]) * bx[j], 700.0))
                    acc_p = np.where(E[rows[idx]] == 1, H * np.exp(1.0 - H), np.exp(-H))
                    acc = rng.uniform(size=len(idx)) < acc_p
                    diag["proposals"] += len(idx)
                    draw[idx[acc]] = prop[acc]; pending[idx[acc]] = False
                X[rows, j] = draw
                diag["draws"] += len(rows); diag["not_accepted"] += int(pending.sum())
                if cyc == n_cycles - 1:
                    diag["final_cycle_not_accepted"] += int(pending.sum())
        out.append(X.copy())
    return (out, diag) if return_diag else out


# ------------------------------------------------------------------ envelope over mechanism parameters
class Envelope:
    """Union of core intervals over configurations of the MNAR shift delta (one coordinate per heavily
    missing variable). `core(delta_vector) -> (est, lo, hi)` arrays over the targets. Results are cached,
    so the different search strategies share their evaluations."""

    def __init__(self, core, q):
        self.core, self.q, self.cache = core, q, {}

    def eval(self, d):
        key = tuple(np.round(np.asarray(d, float), 10))
        if key not in self.cache:
            self.cache[key] = self.core(np.asarray(d, float))
        return self.cache[key]

    def _union(self, keys):
        lo = np.min([self.cache[k][1] for k in keys], 0); hi = np.max([self.cache[k][2] for k in keys], 0)
        return lo, hi

    def vertex_2q3(self, dmax):
        q = self.q; cfg = [np.zeros(q)]
        for j in range(q):
            for s in (1, -1):
                d = np.zeros(q); d[j] = s * dmax; cfg.append(d)
        cfg += [np.full(q, dmax), np.full(q, -dmax)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def adaptive(self, dmax):
        """2q+3 configurations plus the sign-matched vertex for each endpoint of each target."""
        (_, _), _ = self.vertex_2q3(dmax)
        q = self.q
        plus = [self.eval(np.eye(q)[j] * dmax) for j in range(q)]
        minus = [self.eval(-np.eye(q)[j] * dmax) for j in range(q)]
        n_t = len(plus[0][0]) if q else 0
        verts = []
        for t in range(n_t):
            s_lo = np.array([np.sign(plus[j][1][t] - minus[j][1][t]) for j in range(q)])
            s_hi = np.array([np.sign(plus[j][2][t] - minus[j][2][t]) for j in range(q)])
            verts.append(-dmax * np.where(s_lo == 0, 1, s_lo))     # minimises the lower endpoint
            verts.append(dmax * np.where(s_hi == 0, 1, s_hi))      # maximises the upper endpoint
        cfg = [np.zeros(q)] + [np.eye(q)[j] * s * dmax for j in range(q) for s in (1, -1)] + \
              [np.full(q, dmax), np.full(q, -dmax)] + verts
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def vertices(self, dmax):
        """Reference configuration and all 2^q vertices of the box (full vertex enumeration).
        Exact for the interval hull whenever each endpoint is monotone in each coordinate (Proposition 5)."""
        q = self.q
        cfg = [np.zeros(q)] + [np.array(v, float) * dmax for v in itertools.product((-1, 1), repeat=q)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def dense(self, dmax, n_random=100, rng=None):
        """All 2^q vertices plus n_random uniform points in the box (verification only)."""
        q = self.q; rng = rng or np.random.default_rng(0)
        cfg = [np.array(v) * dmax for v in itertools.product((-1, 1), repeat=q)] + \
              [rng.uniform(-dmax, dmax, q) for _ in range(n_random)] + [np.zeros(q)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))


def coverage_summary(df, oracle="Oracle"):
    """Mean / lowest coverage across terms, Monte Carlo SE of the mean, median width relative to oracle."""
    d = df.copy(); d["cover"] = (d.lo <= d.truth) & (d.truth <= d.hi); d["w"] = d.hi - d.lo
    ow = d[d.method == oracle].groupby("term").w.mean()
    g = d.groupby(["method", "term"]).agg(cvg=("cover", "mean"), w=("w", "mean"), R=("cover", "size")).reset_index()
    g["mcse"] = np.sqrt(g.cvg * (1 - g.cvg) / g.R)
    g["wr"] = g.w / g.term.map(ow)
    s = g.groupby("method").agg(mean_coverage=("cvg", "mean"), lowest_coverage=("cvg", "min"),
                                width_vs_oracle=("wr", "median"), reps=("R", "min"))
    return s, g

## 3. Data preparation
Per stay and variable: 48-hour mean (the true value), minimum, maximum, first value and number of measurements. Values of −1 and physiologically impossible zeros are dropped.

In [ ]:
import glob

def build_summary(src):
    frames = []
    for s in ["set-a", "set-b", "set-c"]:
        for f in sorted(glob.glob(os.path.join(src, s, "*.txt"))):
            d = pd.read_csv(f); d["RecordID"] = int(d.loc[d.Parameter == "RecordID", "Value"].iloc[0]); frames.append(d)
    L = pd.concat(frames, ignore_index=True); L = L[L.Parameter != "RecordID"]
    L["Value"] = pd.to_numeric(L.Value, errors="coerce")
    L["hours"] = L.Time.str.split(":").apply(lambda x: int(x[0]) + int(x[1]) / 60)
    desc = ["Age", "Gender", "Height", "ICUType", "Weight"]
    D = L[L.Parameter.isin(desc) & (L.hours == 0)].pivot_table(index="RecordID", columns="Parameter", values="Value", aggfunc="first")
    T = L[~L.Parameter.isin(desc) | (L.hours > 0)].copy(); T = T[T.Value >= 0]
    nonzero = ["HR", "MAP", "NIMAP", "Temp", "BUN", "Creatinine", "HCT", "WBC", "Platelets", "Glucose", "Na", "K", "HCO3", "GCS",
               "RespRate", "pH", "PaO2", "PaCO2", "Albumin", "Bilirubin", "Lactate", "FiO2", "Mg"]
    T = T[~(T.Parameter.isin(nonzero) & (T.Value <= 0))].sort_values(["RecordID", "Parameter", "hours"])
    g = T.groupby(["RecordID", "Parameter"]).Value
    S = pd.concat({"mean": g.mean(), "min": g.min(), "max": g.max(), "first": g.first(), "n": g.size()}, axis=1).unstack("Parameter")
    S.columns = [f"{p}_{stat}" for stat, p in S.columns]
    u = T[T.Parameter == "Urine"].groupby("RecordID").Value.sum().rename("Urine_total")
    mv = T[T.Parameter == "MechVent"].groupby("RecordID").Value.max().rename("MechVent_any")
    O = pd.concat([pd.read_csv(os.path.join(src, f"Outcomes-{k}.txt")) for k in "abc"]).set_index("RecordID")
    X = D.join(S, how="left").join(u).join(mv).join(O); X["MechVent_any"] = X.MechVent_any.fillna(0)
    return X

if not os.path.exists(f"{OUTDIR}/p12_summary.pkl"):
    build_summary(DATA_DIR).to_pickle(f"{OUTDIR}/p12_summary.pkl")

## 4. Analysis code
Variables, full-cohort reference values, scenario B deletion, HPPSM with κ and δ envelopes, the resampling replicate, the dense search, the measurement-error diagnostics and the application with crisp and graded contours.

In [ ]:
import os, sys, glob, time, json
"""
p12_study.py - PhysioNet/CinC Challenge 2012 analyses for Draft 7.

  coverage : real-data resampling experiment with full-cohort reference values (scenarios A and B), 1,000 replicates
  dense    : verification of the finite-configuration union (scenario B: all 16 delta-vertices + 50 random
             points, kappa on a 0.1 grid)
  me       : empirical check of the measurement-error assumptions (bias, bound, nondifferentiality,
             within-patient autocorrelation)
  app      : full-cohort application, with crisp and graded (triangular) contours
"""
import os, sys, glob, time, json, argparse
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

SUMMARY = f"{OUTDIR}/p12_summary.pkl"; RAWDIR = DATA_DIR
VARS = ["HR", "Temp", "GCS", "BUN", "Creatinine", "HCT", "WBC", "Platelets", "Glucose", "Na", "K", "HCO3"]
LOGV = {"BUN", "Creatinine", "WBC", "Platelets", "Glucose"}
PHYS = {"HR": (20, 220), "Temp": (30, 43), "GCS": (3, 15), "BUN": (2, 200), "Creatinine": (0.2, 15), "HCT": (10, 60),
        "WBC": (0.1, 100), "Platelets": (5, 1000), "Glucose": (20, 1000), "Na": (110, 175), "K": (1.5, 8), "HCO3": (5, 50)}
NORMAL = {"BUN": 14, "Creatinine": 0.9, "Glucose": 110, "WBC": 8}; MNAR_VARS = list(NORMAL)
TERMS = VARS + ["Age"]; DMAX = 0.5; KAPPAS = (0.0, 0.5, 1.0)
f = lambda v, x: np.log(x) if v in LOGV else x


def covariates(X):
    Z = pd.DataFrame({"Age": X.Age, "male": (X.Gender == 1).astype(float), "MechVent": X.MechVent_any.astype(float),
                      **{f"ICU{k}": (X.ICUType == k).astype(float) for k in (2, 3, 4)}}).reset_index(drop=True)
    get = lambda st: pd.DataFrame({v: f(v, X[f"{v}_{st}"].clip(*PHYS[v])) for v in VARS}).reset_index(drop=True)
    NN = pd.DataFrame({v: X[f"{v}_n"] for v in VARS}).reset_index(drop=True)
    return Z, get("mean"), get("min"), get("max"), get("first"), NN


RAW = pd.read_pickle(SUMMARY)
keep = RAW[[f"{v}_mean" for v in VARS]].notna().all(1) & RAW.Age.notna() & RAW["In-hospital_death"].notna()
Zall, Mall, LOall, HIall, FIall, NNall = covariates(RAW[keep])
yall = RAW.loc[keep, "In-hospital_death"].values.astype(float)
AGE_J = list(Zall.columns).index("Age")


def logit_terms(y, Xv, Z):
    b, se, _ = logit_fit(y, np.hstack([Xv, Z]))
    idx = list(range(len(VARS))) + [len(VARS) + AGE_J]
    return b[idx], se[idx]


TRUE_COEF = dict(zip(TERMS, logit_terms(yall, Mall.values, Zall.values)[0]))
_b, _, _a = logit_fit(yall, np.hstack([Mall.values, Zall.values]))
LP_TRUE = np.hstack([Mall.values, Zall.values]) @ _b + _a


def mnar_deletion(idx, n, rng):
    sev = (LP_TRUE[idx] - LP_TRUE[idx].mean()) / LP_TRUE[idx].std()
    miss = pd.DataFrame(False, index=range(n), columns=VARS)
    for v in MNAR_VARS:
        a = np.log(0.4 / 0.6)
        for _ in range(30):
            p = 1 / (1 + np.exp(-(a - sev))); a += np.log(0.4 / 0.6) - np.log(p.mean() / (1 - p.mean()))
        miss[v] = rng.uniform(size=n) < 1 / (1 + np.exp(-(a - sev)))
    return miss


def setup(rep, scenario, n=3000):
    rng = np.random.default_rng(50_000 + rep)
    idx = rng.integers(0, len(yall), n)
    y = yall[idx]; Z = Zall.iloc[idx].reset_index(drop=True)
    M, LO, HI, FI, NN = (A.iloc[idx].reset_index(drop=True).copy() for A in (Mall, LOall, HIall, FIall, NNall))
    MID = (LO + HI) / 2
    B2 = pd.DataFrame(error_variance_bound(LO.values, HI.values, NN.values), columns=VARS)
    miss = mnar_deletion(idx, n, rng) if scenario == "B" else pd.DataFrame(False, index=MID.index, columns=VARS)
    return dict(rep=rep, y=y, Z=Z, M=M, MID=MID, FI=FI, B2=B2, miss=miss)


def ar1_design_effect():
    """(1 + rho)/(1 - rho) per variable, rho = median within-patient lag-1 autocorrelation (external information,
    here estimated from the raw PhysioNet series of the whole cohort)."""
    ac = pd.read_csv(f"{OUTDIR}/p12_lag1.csv", index_col=0).iloc[:, 0]
    return np.array([(1 + ac[v]) / (1 - ac[v]) for v in VARS])


def make_core(S, m_env=3, de=None):
    """Returns core(kappa, delta) -> (est, lo, hi) and the list of shifted variables."""
    y, Zv, MID, miss, B2 = S["y"], S["Z"].values, S["MID"], S["miss"], S["B2"].copy()
    if de is not None:
        B2 = B2 * de
    if miss.values.any():
        Xm = MID.mask(miss); sd = Xm.std(); B2 = B2.mask(miss, 0.0)
        imps = [IterativeImputer(random_state=1000 * S["rep"] + k, sample_posterior=True, max_iter=5).fit_transform(
            np.hstack([Zv, y[:, None], Xm.values]))[:, Zv.shape[1] + 1:] for k in range(m_env)]
        shifted = MNAR_VARS
    else:
        imps, sd, shifted = [MID.values], None, []
    su2 = B2.mean().values

    def core(kappa, d):
        ests, vars_ = [], []
        for A in imps:
            A2 = A.copy()
            for j, v in enumerate(shifted):
                if d[j]:
                    A2[miss[v].values, VARS.index(v)] += d[j] * sd[v]
            Xc = regression_calibration(A2, Zv, kappa * su2) if kappa > 0 else A2
            e, s = logit_terms(y, Xc, Zv); ests.append(e); vars_.append(s ** 2)
        return rubin(ests, vars_) if len(imps) > 1 else (np.array(ests[0]), ests[0] - Q95 * np.sqrt(vars_[0]), ests[0] + Q95 * np.sqrt(vars_[0]))
    return core, shifted


def envelope_union(env, dmax, method):
    """method: 'adaptive', '2q3' or 'vertices+axes' (all 2^q vertices, the reference, single-coordinate and
    adaptive configurations; primary in Draft 8). Returns the interval hull of the evaluated core intervals."""
    if method == "adaptive":
        return env.adaptive(dmax)[0]
    if method == "2q3":
        return env.vertex_2q3(dmax)[0]
    (l1, h1), _ = env.vertices(dmax); (l2, h2), _ = env.adaptive(dmax)
    return np.minimum(l1, l2), np.maximum(h1, h2)


def hppsm(S, method="adaptive", kappas=KAPPAS, dmax=DMAX, de=None):
    core, shifted = make_core(S, de=de)
    envs = {k: Envelope(lambda d, k=k: core(k, d), len(shifted)) for k in kappas}
    lo, hi = [], []
    for k, env in envs.items():
        if shifted:
            l, h = envelope_union(env, dmax, method)
        else:
            _, l, h = env.eval(np.zeros(0))
        lo.append(l); hi.append(h)
    est = envs[kappas[0]].eval(np.zeros(len(shifted)))[0]
    return est, np.min(lo, 0), np.max(hi, 0), envs


def wald(e, s):
    return e, e - Q95 * s, e + Q95 * s


def coverage_rep(rep, scenario):
    S = setup(rep, scenario); y, Z, Zv, miss = S["y"], S["Z"], S["Z"].values, S["miss"]
    MIDf, FIf = S["MID"].copy(), S["FI"].copy()
    for v in MNAR_VARS:
        MIDf.loc[miss[v].values, v] = f(v, NORMAL[v]); FIf.loc[miss[v].values, v] = f(v, NORMAL[v])
    out = {"Oracle": wald(*logit_terms(y, S["M"].values, Zv)),
           "Mid-point Wald": wald(*logit_terms(y, MIDf.values, Zv)),
           "First-value Wald": wald(*logit_terms(y, FIf.values, Zv))}
    if scenario == "B":
        out["Missing-indicator Wald"] = wald(*logit_terms(y, MIDf.values, np.hstack([Zv, miss[MNAR_VARS].values.astype(float)])))
        Xm = S["MID"].mask(miss); ests, vars_ = [], []
        for k in range(5):
            A = IterativeImputer(random_state=1000 * rep + k, sample_posterior=True, max_iter=5).fit_transform(
                np.hstack([Zv, y[:, None], Xm.values]))[:, Zv.shape[1] + 1:]
            e, s = logit_terms(y, A, Zv); ests.append(e); vars_.append(s ** 2)
        out["Multiple imputation"] = rubin(ests, vars_)
    e, l, h, envs = hppsm(S, "adaptive")
    out["HPPSM v2 (proposed)"] = (e, l, h)
    if scenario == "B":
        lo = np.min([env.vertex_2q3(DMAX)[0][0] for env in envs.values()], 0)
        hi = np.max([env.vertex_2q3(DMAX)[0][1] for env in envs.values()], 0)
        out["HPPSM v2 (2q+3 configurations)"] = (e, lo, hi)
        for dm in (0.25, 1.0):
            lo = np.min([env.adaptive(dm)[0][0] for env in envs.values()], 0)
            hi = np.max([env.adaptive(dm)[0][1] for env in envs.values()], 0)
            out[f"HPPSM v2 (dmax = {dm})"] = (e, lo, hi)
    # kappa sensitivity: no error correction (kappa = 0 only) and kappa up to 2 (bound exceeded two-fold)
    for kk, nm in (((0.0,), "HPPSM v2 (kappa = 0 only)"), ((0.0, 0.5, 1.0, 1.5, 2.0), "HPPSM v2 (kappa up to 2)")):
        out[nm] = hppsm(S, "adaptive", kappas=kk)[:3]
    out["HPPSM v2 (AR(1)-adjusted bound)"] = hppsm(S, "adaptive", de=ar1_design_effect())[:3]
    return pd.DataFrame([dict(rep=rep, method=k, term=t, est=e_[i], lo=l_[i], hi=h_[i], truth=TRUE_COEF[t])
                         for k, (e_, l_, h_) in out.items() for i, t in enumerate(TERMS)])


def vertex_rep(rep):
    """Scenario B, HPPSM with full vertex enumeration (vertices + axes); data identical to coverage_rep."""
    S = setup(rep, "B")
    e, l, h, _ = hppsm(S, "vertices+axes")
    return pd.DataFrame([dict(rep=rep, method="HPPSM v2 (vertices + axes)", term=t, est=e[i], lo=l[i], hi=h[i], truth=TRUE_COEF[t])
                         for i, t in enumerate(TERMS)])


def dense_rep(rep, n_random=50):
    S = setup(rep, "B"); core, shifted = make_core(S)
    rows = []
    res = {}
    for k in np.round(np.arange(0, 1.01, 0.1), 2):
        env = Envelope(lambda d, k=k: core(k, d), len(shifted))
        if k in KAPPAS:
            res[("adapt", k)] = env.adaptive(DMAX)[0]; res[("2q3", k)] = env.vertex_2q3(DMAX)[0]
            res[("vert", k)] = env.vertices(DMAX)[0]; res[("va", k)] = envelope_union(env, DMAX, "vertices+axes")
        res[("dense", k)] = env.dense(DMAX, n_random, np.random.default_rng(rep))[0]
    U = lambda tag, ks: (np.min([res[(tag, k)][0] for k in ks], 0), np.max([res[(tag, k)][1] for k in ks], 0))
    kd = [k for (t, k) in res if t == "dense"]
    l1, h1 = U("2q3", KAPPAS); l2, h2 = U("adapt", KAPPAS); l3, h3 = U("dense", kd)
    l4, h4 = U("vert", KAPPAS); l5, h5 = U("va", KAPPAS)
    l3, h3 = np.minimum(l3, np.minimum(l1, l2)), np.maximum(h3, np.maximum(h1, h2))
    for i, t in enumerate(TERMS):
        rows.append(dict(rep=rep, term=t, truth=TRUE_COEF[t], lo_2q3=l1[i], hi_2q3=h1[i], lo_adapt=l2[i], hi_adapt=h2[i],
                         lo_vert=l4[i], hi_vert=h4[i], lo_va=l5[i], hi_va=h5[i], lo_full=l3[i], hi_full=h3[i]))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ measurement-error diagnostics
def me_empirical(out):
    """Per variable: standardised mean of U = W - X*, Var(U) relative to the mean bound, correlation of U with
    X*, the log odds ratio of U when added to the oracle model (nondifferentiality), and within-patient lag-1
    autocorrelation of the measurements."""
    U = (LOall + HIall) / 2 - Mall
    B2 = pd.DataFrame(error_variance_bound(LOall.values, HIall.values, NNall.values), columns=VARS)
    rows = []
    base = np.hstack([Mall.values, Zall.values])
    for j, v in enumerate(VARS):
        b, se, _ = logit_fit(yall, np.hstack([base, U[[v]].values]))
        rows.append(dict(variable=v, n_median=float(NNall[v].median()), mean_U_over_sdX=U[v].mean() / Mall[v].std(),
                         ratio_EU2_to_bound=float((U[v] ** 2).mean() / B2[v].mean()),
                         ratio_VarU_to_bound=float(U[v].var() / B2[v].mean()),
                         corr_U_X=float(np.corrcoef(U[v], Mall[v])[0, 1]),
                         reliability_mid=float(1 - U[v].var() / ((LOall[v] + HIall[v]) / 2).var()),
                         logOR_U_per_SD=float(b[-1] * U[v].std()), z_U=float(b[-1] / se[-1])))
    me = pd.DataFrame(rows)
    # within-patient lag-1 autocorrelation from the raw time series (patients with >= 6 measurements)
    cache = os.path.join(out, "p12_lag1.csv")
    if not os.path.exists(cache):
        frames = []
        for s in ["set-a", "set-b", "set-c"]:
            for fn in glob.glob(os.path.join(RAWDIR, s, "*.txt")):
                d = pd.read_csv(fn); d = d[d.Parameter.isin(VARS)]
                d["hours"] = d.Time.str.split(":").apply(lambda x: int(x[0]) + int(x[1]) / 60)
                d = d[(d.hours > 0) & (d.Value > 0)]
                frames.append(d.assign(file=fn))
        L = pd.concat(frames).sort_values(["file", "Parameter", "hours"])
        def lag1(x):
            x = x.values
            if len(x) < 6 or np.std(x) == 0:
                return np.nan
            x = x - x.mean(); return float(np.sum(x[1:] * x[:-1]) / np.sum(x * x))
        ac = L.groupby(["Parameter", "file"]).Value.apply(lag1).groupby("Parameter").median()
        ac.rename("lag1_autocorr_median").to_csv(cache)
    ac = pd.read_csv(cache, index_col=0).iloc[:, 0]
    me["lag1_autocorr_median"] = me.variable.map(ac)
    return me


# ------------------------------------------------------------------ application with crisp and graded contours
def application(out):
    X = RAW[RAW.Age.notna() & RAW["In-hospital_death"].notna()]
    ya = X["In-hospital_death"].values.astype(float)
    Za, MEANa, LOa, HIa, _, NNa = covariates(X); Zav = Za.values
    MIDa = (LOa + HIa) / 2; missa = MIDa.isna()
    B2a = pd.DataFrame(error_variance_bound(LOa.fillna(0).values, HIa.fillna(0).values, NNa.fillna(1).values), columns=VARS)
    sd_a = MIDa.std(); cc = ~missa.any(axis=1).values
    res = {"Oracle (true 48-h means)": wald(*logit_terms(ya[cc], MEANa.values[cc], Zav[cc])),
           "Mid-point Wald": wald(*logit_terms(ya[cc], MIDa.values[cc], Zav[cc]))}
    imps = [IterativeImputer(random_state=k, sample_posterior=True, max_iter=10).fit_transform(
        np.hstack([Zav, ya[:, None], MIDa.values]))[:, Zav.shape[1] + 1:] for k in range(5)]
    heavy = [v for v in VARS if missa[v].mean() > 0.01] or VARS[:4]

    def core(kappa, d, alpha=0.05):
        ests, vars_ = [], []
        for A in imps:
            A2 = A.copy()
            for j, v in enumerate(heavy):
                if d[j]:
                    A2[missa[v].values, VARS.index(v)] += d[j] * sd_a[v]
            Xc = regression_calibration(A2, Zav, kappa * B2a.mean().values) if kappa > 0 else A2
            e, s = logit_terms(ya, Xc, Zav); ests.append(e); vars_.append(s ** 2)
        return rubin(ests, vars_, alpha)
    est0 = core(0.0, np.zeros(len(heavy)))
    res["Multiple imputation"] = est0
    lo, hi = [], []
    for k in KAPPAS:
        # q = 12 shifted variables here, so 2^12 vertices x 3 kappas x 5 imputations is not affordable: adaptive search
        l, h = envelope_union(Envelope(lambda d, k=k: core(k, d), len(heavy)), DMAX, "adaptive"); lo.append(l); hi.append(h)
    res["HPPSM v2 (proposed)"] = (est0[0], np.min(lo, 0), np.max(hi, 0))
    scale = np.array([sd_a[v] for v in VARS] + [X.Age.std()])
    app = pd.DataFrame([dict(method=k, term=t, est=e[i] * scale[i], lo=l[i] * scale[i], hi=h[i] * scale[i])
                        for k, (e, l, h) in res.items() for i, t in enumerate(TERMS)])
    orc = app[app.method.str.startswith("Oracle")].set_index("term").est
    app["contains_oracle"] = (app.lo <= app.term.map(orc)) & (app.term.map(orc) <= app.hi)
    app.to_csv(os.path.join(out, "application.csv"), index=False)
    # graded contour: triangular on delta (sup-norm) and on kappa, pi(psi) = 1 - max(|delta|/dmax, kappa);
    # alpha-cut = union over {psi : pi(psi) >= alpha} of the (1 - alpha) core intervals. Crisp: Psi fixed.
    rows = []
    for a in [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        for kind in ("graded", "crisp"):
            r = (1 - a) if kind == "graded" else 1.0
            lo, hi = [], []
            for k in sorted(set([0.0, 0.5 * r, r])):
                env = Envelope(lambda d, k=k: core(k, d, a), len(heavy))
                l, h = envelope_union(env, DMAX * r, "adaptive")
                lo.append(l); hi.append(h)
            l, h = np.min(lo, 0), np.max(hi, 0)
            for i, t in enumerate(TERMS):
                rows.append(dict(alpha=a, contour=kind, term=t, lo=l[i] * scale[i], hi=h[i] * scale[i]))
        cfg = core(0.0, np.zeros(len(heavy)), a)
        for i, t in enumerate(TERMS):
            rows.append(dict(alpha=a, contour="MI only", term=t, lo=cfg[1][i] * scale[i], hi=cfg[2][i] * scale[i]))
    pd.DataFrame(rows).to_csv(os.path.join(out, "application_contours.csv"), index=False)
    return app



import time
def run_resumable(path, reps, fn, key="rep", start=0):
    """Append one replicate at a time to `path`; replicates already present are skipped."""
    done = set(pd.read_csv(path)[key]) if os.path.exists(path) else set()
    for r in range(start, reps):
        if r in done:
            continue
        t0 = time.time(); res = fn(r)
        rows, info = res if isinstance(res, tuple) else (res, None)
        rows.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        if info is not None:
            ip = path.replace(".csv", "_info.csv"); pd.DataFrame([info]).to_csv(ip, mode="a", header=not os.path.exists(ip), index=False)
        print(f"{os.path.basename(path)}: replicate {r} ({time.time()-t0:.0f} s)")

def summary_table(d, order=None, oracle="Oracle"):
    s, g = coverage_summary(d.drop_duplicates(["rep", "method", "term"]), oracle)
    s["mcse_of_mean"] = g.groupby("method").mcse.mean()
    return s.reindex(order) if order else s

## 5. Measurement-error diagnostics (Table 5)

In [ ]:
me_tab = me_empirical(OUTDIR); me_tab.to_csv(f"{OUTDIR}/me_empirical.csv", index=False); me_tab.round(3)

## 6. Synthetic measurement-error simulation (Fig. 4)
108 scenarios: AR(1) correlation, number of measurements, Gaussian or skewed within-patient variation, constant or level-dependent spread, and fixed, level-dependent or outcome-dependent frequency.

In [ ]:
"""
me_sim.py - when does the error-variance bound (1) and regression calibration work?

Each patient i has a true level mu_i ~ N(0, 1) and an exact covariate Z_i (corr 0.4 with mu_i). n_i measurements
x_it = mu_i + s_i e_it are taken, where e_it is a stationary AR(1) series with lag-1 correlation rho and unit
variance, either Gaussian or skewed (standardised log-normal, shape 0.75), and s_i = 1 (constant spread) or
exp(0.4 mu_i)/c (spread increasing with level). The number of measurements is n0 (fixed), n0 exp(0.5 mu_i)
(sicker patients measured more) or n0 (1 + Y_i) (outcome-dependent measurement frequency).
The true covariate is X*_i = mean of the n_i measurements (as in PhysioNet), the outcome is
logit P(Y=1) = a + 0.5 X*_i/sd(X*) + 0.3 Z_i (prevalence about 15%), and the analyst sees min, max and n_i.
Methods: oracle, mid-point Wald, regression calibration with kappa = 1, HPPSM (kappa in {0, 0.5, 1}), HPPSM
with kappa up to 2, and HPPSM with the AR(1)-adjusted bound, kappa in [0, (1 + rho)/(1 - rho)]. Estimand: the coefficient of X*.
"""
import os, sys, itertools, time
import numpy as np, pandas as pd

N = 3000; BETA = 0.5


def series(rng, n, nmax, rho, dist):
    e = np.empty((n, nmax)); e[:, 0] = rng.standard_normal(n)
    for t in range(1, nmax):
        e[:, t] = rho * e[:, t - 1] + np.sqrt(1 - rho ** 2) * rng.standard_normal(n)
    if dist == "skewed":
        s = 0.75; e = (np.exp(s * e) - np.exp(s ** 2 / 2)) / np.sqrt((np.exp(s ** 2) - 1) * np.exp(s ** 2))
    return e


def one_rep(rng, rho, n0, dist, spread, freq):
    mu = rng.standard_normal(N); Z = 0.4 * mu + np.sqrt(1 - 0.16) * rng.standard_normal(N)
    s = np.ones(N) if spread == "constant" else np.exp(0.4 * mu) / np.exp(0.08)
    y0 = None
    if freq == "fixed":
        n = np.full(N, n0)
    elif freq == "level":
        n = np.clip(np.round(n0 * np.exp(0.5 * mu)), 3, 4 * n0).astype(int)
    nmax_guess = 2 * n0 if freq == "outcome" else int(n.max()) if freq != "outcome" else 2 * n0
    e = series(rng, N, max(nmax_guess, 3), rho, dist)
    x = mu[:, None] + s[:, None] * e
    if freq == "outcome":
        # outcome generated from the mean of the first n0 measurements, then patients who die get n0 more
        xs0 = x[:, :n0].mean(1); lp0 = BETA * (xs0 - xs0.mean()) / xs0.std() + 0.3 * Z
        a = np.log(0.15 / 0.85) - 0.1; y0 = rng.uniform(size=N) < 1 / (1 + np.exp(-(a + lp0)))
        n = np.where(y0, 2 * n0, n0)
    mask = np.arange(x.shape[1])[None, :] < n[:, None]
    xm = np.where(mask, x, np.nan)
    Xs = np.nanmean(xm, 1); lo = np.nanmin(xm, 1); hi = np.nanmax(xm, 1); W = (lo + hi) / 2
    if y0 is None:
        lp = BETA * (Xs - Xs.mean()) / Xs.std() + 0.3 * Z
        a = np.log(0.15 / 0.85) - 0.1
        y = (rng.uniform(size=N) < 1 / (1 + np.exp(-(a + lp)))).astype(float)
        truth = BETA / Xs.std()
    else:
        y = y0.astype(float); truth = BETA / xs0.std()      # outcome depends on the first n0 measurements
        Xs = xs0                                             # target covariate: mean over the base window
    b2 = error_variance_bound(lo, hi, n); U = W - Xs
    Zc = Z[:, None]
    res = {}
    b, se, _ = logit_fit(y, np.column_stack([Xs, Z])); res["Oracle"] = (b[0], b[0] - Q95 * se[0], b[0] + Q95 * se[0])
    b, se, _ = logit_fit(y, np.column_stack([W, Z])); res["Mid-point Wald"] = (b[0], b[0] - Q95 * se[0], b[0] + Q95 * se[0])
    ints = {}
    for k in (0.0, 0.5, 1.0, 1.5, 2.0):
        Xc = regression_calibration(W[:, None], Zc, np.array([k * b2.mean()]))[:, 0] if k > 0 else W
        b, se, _ = logit_fit(y, np.column_stack([Xc, Z])); ints[k] = (b[0], b[0] - Q95 * se[0], b[0] + Q95 * se[0])
    res["Regression calibration (kappa = 1)"] = ints[1.0]
    res["HPPSM (kappa in [0, 1])"] = (ints[0][0], min(ints[k][1] for k in (0, .5, 1)), max(ints[k][2] for k in (0, .5, 1)))
    res["HPPSM (kappa in [0, 2])"] = (ints[0][0], min(v[1] for v in ints.values()), max(v[2] for v in ints.values()))
    de = (1 + rho) / (1 - rho)                     # AR(1) design effect for a mean: bound (1) x de
    for k in (0.5 * de, de):
        Xc = regression_calibration(W[:, None], Zc, np.array([k * b2.mean()]))[:, 0]
        b, se, _ = logit_fit(y, np.column_stack([Xc, Z])); ints[("de", k)] = (b[0], b[0] - Q95 * se[0], b[0] + Q95 * se[0])
    keys = [0.0, ("de", 0.5 * de), ("de", de)]
    res["HPPSM (AR(1)-adjusted bound)"] = (ints[0][0], min(ints[k][1] for k in keys), max(ints[k][2] for k in keys))
    diag = dict(ratio_EU2_bound=float(np.mean(U ** 2) / np.mean(b2)), meanU_sdX=float(U.mean() / Xs.std()),
                corrUX=float(np.corrcoef(U, Xs)[0, 1]), reliability=float(1 - U.var() / W.var()))
    return res, truth, diag


def main(reps=250, out="results_me", grid_filter=None):
    os.makedirs(out, exist_ok=True)
    path = f"{out}/me_sim.csv"
    done = set(map(tuple, pd.read_csv(path)[["rho", "n0", "dist", "spread", "freq"]].drop_duplicates().values)) if os.path.exists(path) else set()
    grid = list(itertools.product((0.0, 0.5, 0.9), (6, 24, 72), ("normal", "skewed"), ("constant", "level"), ("fixed", "level", "outcome")))
    for gi, g in enumerate(grid):
        if grid_filter is not None and not grid_filter(g):
            continue
        if (g[0], g[1], g[2], g[3], g[4]) in done:
            continue
        rho, n0, dist, spread, freq = g
        rng = np.random.default_rng(880_000 + gi); rows = []; t0 = time.time()
        for r in range(reps):
            res, truth, diag = one_rep(rng, rho, n0, dist, spread, freq)
            for k, (e, l, h) in res.items():
                rows.append(dict(rho=rho, n0=n0, dist=dist, spread=spread, freq=freq, rep=r, method=k, est=e, lo=l, hi=h,
                                 truth=truth, **diag))
        pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        print(g, f"{time.time()-t0:.0f}s", flush=True)



main(ME_REPS, out=f"{OUTDIR}/me_sim")
ms = pd.read_csv(f"{OUTDIR}/me_sim/me_sim.csv"); ms["c"] = (ms.lo <= ms.truth) & (ms.truth <= ms.hi)
ms[ms.freq == "fixed"].groupby(["dist", "spread", "rho", "n0", "method"]).c.mean().unstack("method").round(2)

## 7. Resampling experiment (Table 6, Fig. 2 right)

In [ ]:
for sc in "AB":
    run_resumable(f"{OUTDIR}/coverage_{sc}_0.csv", COV_REPS, lambda r, sc=sc: coverage_rep(r, sc))
run_resumable(f"{OUTDIR}/vertex_B_0.csv", COV_REPS, vertex_rep)      # primary for B: vertices + axes
REN = {"HPPSM v2 (proposed)": "HPPSM v2 (adaptive)", "HPPSM v2 (vertices + axes)": "HPPSM v2 (proposed)"}
def load_p12(sc):
    d = pd.read_csv(f"{OUTDIR}/coverage_{sc}_0.csv").drop_duplicates(["rep", "method", "term"])
    if sc == "B":
        v = pd.read_csv(f"{OUTDIR}/vertex_B_0.csv"); d = pd.concat([d, v[v.rep.isin(d.rep)]]).assign(method=lambda x: x.method.replace(REN))
    return d
ORDER = ["Oracle", "Mid-point Wald", "First-value Wald", "Missing-indicator Wald", "Multiple imputation", "HPPSM v2 (kappa = 0 only)",
         "HPPSM v2 (2q+3 configurations)", "HPPSM v2 (adaptive)", "HPPSM v2 (proposed)", "HPPSM v2 (kappa up to 2)", "HPPSM v2 (AR(1)-adjusted bound)",
         "HPPSM v2 (dmax = 0.25)", "HPPSM v2 (dmax = 1.0)"]
table6 = pd.DataFrame({sc: summary_table(load_p12(sc), ORDER).apply(
    lambda r: f"{100*r.mean_coverage:.1f}% ({100*r.lowest_coverage:.0f}%) · {r.width_vs_oracle:.2f}×" if pd.notna(r.mean_coverage) else "–", axis=1) for sc in "AB"})
table6.to_csv(f"{OUTDIR}/table6.csv"); table6

## 8. Dense search, scenario B (Table 4)

In [ ]:
run_resumable(f"{OUTDIR}/dense_B_0.csv", DENSE_REPS, dense_rep)
d = pd.read_csv(f"{OUTDIR}/dense_B_0.csv"); w = d.hi_full - d.lo_full
for tag in ("2q3", "adapt", "va"):
    sh = (np.maximum(d[f"lo_{tag}"] - d.lo_full, 0) + np.maximum(d.hi_full - d[f"hi_{tag}"], 0)) / w
    print(tag, f"mean shortfall {100*sh.mean():.2f}%, max {100*sh.max():.1f}%, coverage {((d[f'lo_{tag}'] <= d.truth) & (d.truth <= d[f'hi_{tag}'])).mean():.3f}")

## 9. Application with crisp and graded contours (Table 7, Figs. 5–6)

In [ ]:
app = application(OUTDIR)
print(app.assign(ci=app.apply(lambda r: f"{r.est:+.2f} ({r.lo:+.2f}, {r.hi:+.2f})", axis=1)).pivot(index="term", columns="method", values="ci").to_string())
import matplotlib.pyplot as plt
ct = pd.read_csv(f"{OUTDIR}/application_contours.csv"); orc = app[app.method.str.startswith("Oracle")].set_index("term").est
fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), sharey=True)
for ax, t in zip(axes, ["Temp", "BUN", "HCT", "GCS"]):
    for kind, (c_, ls) in {"graded": ("#E67E22", "-"), "crisp": ("#A04000", "--"), "MI only": ("#2E86C1", ":")}.items():
        g = ct[(ct.term == t) & (ct.contour == kind)].sort_values("alpha")
        ax.plot(g.lo, g.alpha, ls, color=c_, label=kind); ax.plot(g.hi, g.alpha, ls, color=c_)
    ax.axhline(0.05, color="g", ls="--", lw=0.8); ax.plot([orc[t]], [0], "k|", ms=14, mew=2, clip_on=False); ax.set_title(t)
axes[0].set_ylabel("alpha"); axes[0].legend(fontsize=7)
plt.savefig(f"{OUTDIR}/fig6_graded.png", dpi=300, bbox_inches="tight"); plt.show()

## 10. Discrimination (AUC = C-index for a binary outcome; Fig. 7)

In [ ]:
from sklearn.metrics import roc_auc_score

def cindex_rep(rep, sc, n):
    rng = np.random.default_rng(700_000 + 1000 * rep + n + (1 if sc == "A" else 2))
    perm = rng.permutation(len(yall)); tr, te = perm[:n], perm[n:]
    y = yall[tr]; Z = Zall.values[tr]; M_ = Mall.values[tr]; MID = ((LOall + HIall) / 2).values[tr].copy(); FI = FIall.values[tr].copy()
    miss = np.zeros_like(MID, bool)
    if sc == "B":
        sev = (LP_TRUE[tr] - LP_TRUE[tr].mean()) / LP_TRUE[tr].std()
        for v in MNAR_VARS:
            j = VARS.index(v); a = np.log(0.4 / 0.6)
            for _ in range(30):
                p = 1 / (1 + np.exp(-(a - sev))); a += np.log(0.4 / 0.6) - np.log(p.mean() / (1 - p.mean()))
            miss[:, j] = rng.uniform(size=n) < 1 / (1 + np.exp(-(a - sev)))
            MID[miss[:, j], j] = f(v, NORMAL[v]); FI[miss[:, j], j] = f(v, NORMAL[v])
    Xte = np.hstack([Mall.values[te], Zall.values[te]]); yte = yall[te]
    auc = lambda b: roc_auc_score(yte, Xte @ b)
    res = {"Oracle": auc(logit_fit(y, np.hstack([M_, Z]))[0]), "Mid-point": auc(logit_fit(y, np.hstack([MID, Z]))[0]),
           "First value": auc(logit_fit(y, np.hstack([FI, Z]))[0])}
    if sc == "B":
        Xm = MID.copy(); Xm[miss] = np.nan
        res["HPPSM v2 (proposed)"] = auc(np.mean([logit_fit(y, np.hstack([IterativeImputer(random_state=k, sample_posterior=True, max_iter=5).fit_transform(
            np.hstack([Z, y[:, None], Xm]))[:, Z.shape[1] + 1:], Z]))[0] for k in range(3)], 0))
    else:
        res["HPPSM v2 (proposed)"] = res["Mid-point"]
    return pd.DataFrame([dict(scenario=sc, n=n, rep=rep, method=k, C=v) for k, v in res.items()])

path = f"{OUTDIR}/cindex.csv"
done = set(map(tuple, pd.read_csv(path)[["scenario", "n", "rep"]].values)) if os.path.exists(path) else set()
for rep in range(CIDX_REPS):
    for sc in "AB":
        for n in (500, 1000, 3000):
            if (sc, n, rep) not in done:
                cindex_rep(rep, sc, n).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
pd.read_csv(path).groupby(["scenario", "method", "n"]).C.mean().unstack("n").round(3)

## Session information

In [ ]:
import sys, scipy, sklearn
print(sys.version); print({m.__name__: m.__version__ for m in (np, pd, scipy, sklearn)})